# Structural audit — cặp Qwen3-Embedding-0.6B → MiniLMv2-L6-H384 (run)

Notebook này **chạy** các thực nghiệm của `docs/experiments_and_figures.md` cho cặp
nhỏ nhất của bài (student 384-d, 6 layer) — đúng "Week 1" trong §5 của protocol: mọi
arm ở một seed để ra quyết định go/no-go, rồi thêm seed cho các arm mang claim. Nó
được viết theo khuôn của `main_tables.ipynb` (cùng cách clone repo, cùng cache
teacher, cùng cách chấm điểm), nhưng khác hai chỗ:

1. **Prerequisites §0** được làm ngay trong notebook: corpus được **dedup** khỏi mọi
   câu evaluation (§0.2, có manifest), một **probe set** cố định được dựng một lần
   (§0.1), và teacher được encode trên probe set một lần cho cả cặp.
2. Mỗi arm chạy ở **matched HP** (§0.3: một lr, một batch, một số epoch, AdamW, một
   λ cho mọi arm), lưu weight student **mỗi epoch**, và ngay sau khi xong, student ở
   từng epoch được encode trên probe set thành `probe_final.pt` / `probe_layers.pt`
   / `train_summary.json` trong thư mục arm. Mọi rung của ladder, mọi depth profile
   và mọi figure sau đó là tính post-hoc trong `audit_analysis_qwen_minilm.ipynb`
   — đổi metric không bao giờ phải train lại.

Arm được nhóm theo claim. Cột **trạng thái** nói arm đó chạy được bằng flag hiện có
hay còn thiếu code; arm thiếu code vẫn nằm trong plan (để bảng tổng hợp nói rõ nó
chưa có) nhưng bị bỏ qua khi chạy.

| Claim | Arm | Trạng thái |
|---|---|---|
| C1 interface | `pca__procrustes` (ours), `pca__none`, `pca__random` ×draws, `random__none` ×draws, `mrl_prefix__none`, `learned_t2s` / `learned_s2t` (lr ×1, ×5), `pca__mse`, `simcse_only` | chạy được |
| C1 interface | `procrustes_per_batch`, `pca_k{64,128,256}` (Figure 7) | thiếu code |
| C2 structure | `+gram`, `+lasd`, `+anchor_k` | thiếu code |
| C3 depth | `freeze_lower` | thiếu code |
| G gauge | null band = `pca__random` ×draws (chạy được); `gauge_theta{¼,½,¾}`, `gauge_rank_one` | thiếu code |
| M main | `talas__matched` (SAM như code gốc), `ours__talas15k`, corpus sweep | chạy được |
| M main | `talas__adamw` | thiếu code |

Bảng main results đầy đủ (8 method) vẫn là việc của `main_tables.ipynb`; notebook này
chỉ thêm các arm mà protocol yêu cầu thêm.

**Dung lượng.** Mỗi arm giữ `weights/student_epoch_N.pt` (~90 MB × 5) và hai file
probe (~100 MB); checkpoint đầy đủ (có optimizer, ~270 MB × 5) bị cắt bớt theo
`KEEP_FULL_CHECKPOINTS` sau khi đã dump probe. Với ~20 arm cần khoảng 12 GB.


In [ ]:
# 1. Cấu hình thí nghiệm. Chỉnh các giá trị trong cell này trước khi chạy.
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"

# Cặp của notebook này. Giữ cùng cấu trúc với PAIRS của main_tables.ipynb để hai
# notebook đọc giống nhau; chỉ có một entry vì audit chạy trên cặp nhỏ nhất trước.
PAIRS = {
    "qwen3_0.6b_to_minilm_h384": {
        "teacher": "Qwen/Qwen3-Embedding-0.6B",
        "student": "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base",
        "teacher_pooling": "last_token",
        "teacher_special_token": "Ġ",
        "emo_teacher_special_token": None,
        "min_vram_gib": 12,
        "note": "student nhỏ (384-d, 6 layer): cặp go/no-go của protocol",
    },
}
PAIR = "qwen3_0.6b_to_minilm_h384"
PAIR_CONFIG = PAIRS[PAIR]
TEACHER_MODEL = PAIR_CONFIG["teacher"]
STUDENT_MODEL = PAIR_CONFIG["student"]
TEACHER_POOLING = PAIR_CONFIG["teacher_pooling"]
# Pooling của student: geoode/simcse/talas đều supervise và chấm điểm vector CLS.
STUDENT_POOLING = "cls"

DATASETS = {
    "talas_15k": {
        "path": Path("data/train_set/merged_3_data_5k_each.csv"),
        "build": None,
        "note": "setup của bài TALAS: ~15k câu từ EMOTION/WiC/STS-B",
    },
    "100k": {
        "path": Path("data/train_set/train_100k.csv"),
        "build": None,
        "note": "102,361 dòng benchmark train — KHÔNG phải base 100k của 150k/200k",
    },
    "150k": {
        "path": Path("data/train_set/train_150k.csv"),
        "build": "scripts/build_train_corpus.py --total 150000",
        "note": "100k base + 25k MS MARCO query + 25k MS MARCO passage",
    },
    "200k": {
        "path": Path("data/train_set/train_200k.csv"),
        "build": "scripts/build_train_corpus.py --total 200000",
        "note": "100k base + 50k MS MARCO query + 50k MS MARCO passage",
    },
}
DATASET = "100k"
# §0.2: bỏ khỏi corpus mọi câu xuất hiện trong bất kỳ split evaluation nào (exact
# match sau khi casefold + gộp whitespace). File dedup nằm cạnh file gốc, kèm
# manifest. Cache teacher khoá theo *nội dung* corpus nên corpus dedup có cache riêng.
DEDUP_CORPUS = True
# Corpus nhỏ của TALAS cho arm "ours trên setting của TALAS" (§0.3 b).
TALAS_SMALL_DATASET = "talas_15k"
# Corpus-size sweep (§M): ours + baseline mạnh nhất ở từng mức. Rỗng = không chạy.
# Mỗi mức cần cache teacher riêng, nên chỉ bật khi đã xong phần chính.
CORPUS_SWEEP_DATASETS = []

# §0.3 matched HP: một bộ cho MỌI arm và MỌI baseline. lr 5e-5 / batch 64 là bộ
# đã tune cho geoode ở main_tables.ipynb; TALAS ở bảng chính chạy 2e-5, ở đây nó
# chạy cùng 5e-5 để so sánh matched (dòng TALAS-2e-5 vẫn nằm ở main_tables).
MATCHED_HP = {
    "batch_size": 64,
    "learning_rate": 5e-5,
    "epochs": 5,
    "lambda_ctr": 0.5,
    "max_length": 256,
}
EPOCHS = MATCHED_HP["epochs"]
# Seed train. Go/no-go: [42]. Claim: [42, 43, 44] (mỗi arm chạy ở mọi seed).
SEEDS = [42]
# Số draw Haar của arm ngẫu nhiên (random subspace, random gauge): độ tản giữa các
# draw là null band mà PCA và Procrustes phải vượt qua (§4). >= 3.
DRAWS = 3
# lr của map học (learned_t2s / learned_s2t) theo bội số lr student: baseline phải
# được quét ít nhất hai mức để không bị dựng thành bù nhìn.
LEARNED_LR_SCALES = [1.0, 5.0]
# Bật/tắt từng nhóm claim.
GROUPS = {"C1": True, "C2": True, "C3": True, "G": True, "M": True}

# Đánh giá: giống main_tables — eval test ở cuối, không eval từng epoch (probe dump
# đã cho mọi quỹ đạo cần thiết). Retrieval tắt mặc định vì đắt (~92k document/arm).
EVAL_ON_TEST_EACH_EPOCH = True
PAIR_THRESHOLD_SOURCE = "test" if EVAL_ON_TEST_EACH_EPOCH else "validation"
EVAL_EVERY = 0
EVAL_RETRIEVAL = False

# §0.1 probe set (dựng một lần, seeded, dùng chung cho mọi arm).
PROBE_SEED = 0
PROBE_CORPUS_SENTENCES = 4096      # câu corpus
PROBE_DOCS_PER_RETRIEVAL = 4096    # document mỗi benchmark retrieval
PROBE_EVAL_SPLITS = ("test_set",)  # câu evaluation: split test của 9 task
PROBE_CORE_EVAL = 4096             # phần "core" (dump mọi layer, mọi epoch)
PROBE_CORE_RETRIEVAL = 1024
PROBE_MAX_LENGTH = 256
PROBE_BATCH_SIZE = 256
# Dump per-layer ở mọi epoch (Figure 4 cần cột = epoch). False = chỉ epoch cuối.
PROBE_LAYERS_EVERY_EPOCH = True
# Checkpoint đầy đủ (có optimizer state) giữ lại sau khi dump: "all" | "last" | "none".
# Weight student từng epoch (weights/) và probe dump luôn được giữ.
KEEP_FULL_CHECKPOINTS = "last"

NUM_WORKERS = 2
CUDA_VISIBLE_DEVICES = "0,1"
STOP_ON_ERROR = True
AUTO_FETCH_DATA = True
SAVE_TO_GOOGLE_DRIVE = False

RUN_STAMP = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M%S")
DATASET_TAG = f"{DATASET}_dedup" if DEDUP_CORPUS else DATASET
RUN_NAME = f"audit_{PAIR}_{DATASET_TAG}_{RUN_STAMP}"
# Đặt RUN_NAME thành tên một run cũ để chạy tiếp (arm đã có final test bị bỏ qua).

print(f"Pair: {PAIR} — {PAIR_CONFIG['note']}")
print(f"  teacher: {TEACHER_MODEL} (pooling={TEACHER_POOLING})")
print(f"  student: {STUDENT_MODEL} (pooling={STUDENT_POOLING})")
print(f"Dataset: {DATASET_TAG} — {DATASETS[DATASET]['note']}")
print(f"Matched HP: {MATCHED_HP}; seeds={SEEDS}; draws={DRAWS}; learned lr x{LEARNED_LR_SCALES}")
print(f"Run name: {RUN_NAME}")


In [ ]:
# 2. Dùng repo hiện tại nếu notebook nằm trong repo; nếu không thì clone từ GitHub.
import subprocess
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
PROJECT_DIR = next(
    (p for p in candidates if (p / "main.py").is_file() and (p / "distiller.py").is_file()),
    None,
)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if PROJECT_DIR.exists():
        assert (PROJECT_DIR / "main.py").is_file(), (
            f"Thư mục đã tồn tại nhưng không phải repo hợp lệ: {PROJECT_DIR}"
        )
        print(f"Reuse existing clone: {PROJECT_DIR}")
    else:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

assert (PROJECT_DIR / "requirements.txt").is_file()
print(f"Project directory: {PROJECT_DIR}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Dependencies installed.")


In [ ]:
# 3. Nơi lưu output, kiểm tra GPU/data, dựng corpus dedup (§0.2).
import json
import os

import torch

from src.probe_set import deduplicate_corpus

try:
    from google.colab import drive as colab_drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
if IN_COLAB and SAVE_TO_GOOGLE_DRIVE:
    colab_drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/embedding-kd-runs")
else:
    OUTPUT_BASE = PROJECT_DIR / "runs"

RUN_ROOT = OUTPUT_BASE / RUN_NAME
RETRIEVAL_DIR = PROJECT_DIR / "data" / "test_set" / "retrieval"


def run_script(argv, what):
    """Chạy một script trong repo và dừng hẳn nếu nó fail."""
    print(f"[data] {what}: python3 {' '.join(argv)}")
    subprocess.run([sys.executable, *argv], cwd=PROJECT_DIR, check=True)


for split in ("train_set", "val_set", "test_set"):
    split_dir = PROJECT_DIR / "data" / split
    assert split_dir.is_dir() and any(split_dir.glob("*.csv")), f"Thiếu dữ liệu evaluation: {split_dir}"

# Ba benchmark retrieval là exclusion set của corpus builder, nguồn query/document
# của probe set, và (nếu bật) một family của bảng điểm — nên luôn cần có.
missing_retrieval = [
    name for name in ("arguana", "fiqa", "scidocs")
    if not (RETRIEVAL_DIR / name / "corpus.csv").is_file()
]
if missing_retrieval:
    if not AUTO_FETCH_DATA:
        raise FileNotFoundError(
            f"Thiếu benchmark retrieval {missing_retrieval}. "
            "Chạy: python3 scripts/download_retrieval_benchmarks.py"
        )
    run_script(["scripts/download_retrieval_benchmarks.py"], "tải benchmark retrieval")

# Mọi corpus mà plan sẽ đụng tới: corpus chính, corpus nhỏ của TALAS, các mức sweep.
NEEDED_DATASETS = [DATASET]
if GROUPS["M"]:
    NEEDED_DATASETS.append(TALAS_SMALL_DATASET)
    NEEDED_DATASETS.extend(CORPUS_SWEEP_DATASETS)
NEEDED_DATASETS = list(dict.fromkeys(NEEDED_DATASETS))

TRAIN_DATA_BY_DATASET = {}
DEDUP_MANIFESTS = {}
for key in NEEDED_DATASETS:
    spec = DATASETS[key]
    raw = PROJECT_DIR / spec["path"]
    if not raw.is_file():
        if spec["build"] is None:
            raise FileNotFoundError(f"Không tìm thấy corpus {key}: {raw} (lẽ ra nằm sẵn trong repo)")
        if not AUTO_FETCH_DATA:
            raise FileNotFoundError(f"Không tìm thấy corpus {key}: {raw}. Dựng bằng: python3 {spec['build']}")
        run_script(spec["build"].split(), f"dựng corpus {key}")
    if not DEDUP_CORPUS:
        TRAIN_DATA_BY_DATASET[key] = raw
        continue
    dedup = raw.with_name(f"{raw.stem}_dedup.csv")
    manifest_path = dedup.with_suffix(".manifest.json")
    if dedup.is_file() and manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        print(f"[dedup] dùng lại {dedup.name}: {manifest['rows_after']}/{manifest['rows_before']} dòng")
    else:
        manifest = deduplicate_corpus(raw, dedup, PROJECT_DIR, splits=("val_set", "test_set"))
        print(
            f"[dedup] {raw.name} -> {dedup.name}: bỏ {manifest['rows_removed']} / "
            f"{manifest['rows_before']} dòng trùng câu evaluation "
            f"(theo task: {manifest['removed_by_task']})"
        )
    TRAIN_DATA_BY_DATASET[key] = dedup
    DEDUP_MANIFESTS[key] = manifest
TRAIN_DATA = TRAIN_DATA_BY_DATASET[DATASET]

if not torch.cuda.is_available():
    raise RuntimeError(f"{TEACHER_MODEL} cần GPU; hãy bật GPU runtime rồi chạy lại.")
if hasattr(torch.cuda, "is_bf16_supported") and not torch.cuda.is_bf16_supported():
    raise RuntimeError("Config của repo tải teacher bằng BF16 nhưng GPU hiện tại không hỗ trợ BF16.")

run_root_existed = RUN_ROOT.is_dir()
RUN_ROOT.mkdir(parents=True, exist_ok=True)
if run_root_existed:
    print(f"[note] RUN_ROOT đã tồn tại, sẽ chạy tiếp vào run cũ: {RUN_ROOT}")
print(f"PyTorch: {torch.__version__}; CUDA build: {torch.version.cuda}")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"  cuda:{index}: {props.name} ({props.total_memory / 2**30:.1f} GiB)")
train_rows = sum(1 for _ in TRAIN_DATA.open(encoding="utf-8")) - 1
print(f"Training data: {TRAIN_DATA} ({train_rows} dòng)")
print(f"Output root: {RUN_ROOT}")


In [ ]:
# 4. Probe set (§0.1) + teacher và student-init trên probe set — dựng một lần cho cả cặp.
#
# Probe set phụ thuộc vào corpus (4096 câu mẫu) và seed, không phụ thuộc vào arm hay
# run, nên nó nằm ngoài RUN_ROOT và được dùng lại bởi mọi run của cặp. Teacher trên
# probe set cũng vậy (encode một lần, ~55k câu). Student *chưa train* trên probe set
# là trạng thái step-0 chung của mọi arm.
import hashlib
import re

import pandas as pd
from transformers import AutoModel, AutoTokenizer

from src.probe_set import build_probe_set, probe_digest
from src.structural_audit import encode_texts, load_student

PROBE_DIR = OUTPUT_BASE / "probe" / PAIR
PROBE_DIR.mkdir(parents=True, exist_ok=True)
PROBE_PATH = PROBE_DIR / f"probe_{DATASET_TAG}_seed{PROBE_SEED}.csv"

if PROBE_PATH.is_file():
    PROBE = pd.read_csv(PROBE_PATH, keep_default_na=False)
    print(f"[probe] dùng lại {PROBE_PATH.name}: {len(PROBE)} câu")
else:
    PROBE = build_probe_set(
        PROJECT_DIR,
        TRAIN_DATA,
        n_corpus=PROBE_CORPUS_SENTENCES,
        n_docs_per_retrieval=PROBE_DOCS_PER_RETRIEVAL,
        eval_splits=PROBE_EVAL_SPLITS,
        core_eval=PROBE_CORE_EVAL,
        core_retrieval=PROBE_CORE_RETRIEVAL,
        seed=PROBE_SEED,
    )
    PROBE.to_csv(PROBE_PATH, index=False)
    print(f"[probe] dựng {PROBE_PATH.name}: {len(PROBE)} câu")
PROBE_DIGEST = probe_digest(PROBE)
PROBE_TEXTS = PROBE["text"].astype(str).tolist()
CORE_INDEX = PROBE.index[PROBE["core"].astype(bool)].to_numpy()
PROBE_CORE_TEXTS = [PROBE_TEXTS[i] for i in CORE_INDEX]
print(PROBE["group"].value_counts().to_string())
print(f"core: {len(CORE_INDEX)} câu; digest {PROBE_DIGEST}")
(PROBE_DIR / f"probe_{DATASET_TAG}_seed{PROBE_SEED}.manifest.json").write_text(
    json.dumps(
        {
            "probe_digest": PROBE_DIGEST,
            "rows": int(len(PROBE)),
            "core_rows": int(len(CORE_INDEX)),
            "groups": {k: int(v) for k, v in PROBE["group"].value_counts().items()},
            "corpus": str(TRAIN_DATA),
            "seed": PROBE_SEED,
            "eval_splits": list(PROBE_EVAL_SPLITS),
        },
        indent=2,
    ),
    encoding="utf-8",
)


def slug(name):
    return re.sub(r"[^A-Za-z0-9]+", "-", name).strip("-").lower()


DEVICE = torch.device("cuda")
PROBE_TEACHER_PATH = PROBE_DIR / f"teacher_{slug(TEACHER_MODEL)}_{TEACHER_POOLING}_{PROBE_DIGEST}.pt"
if PROBE_TEACHER_PATH.is_file():
    print(f"[probe] teacher đã có: {PROBE_TEACHER_PATH.name}")
else:
    print(f"[probe] encode teacher {TEACHER_MODEL} trên {len(PROBE_TEXTS)} câu...")
    import transformers

    tok_teacher = AutoTokenizer.from_pretrained(TEACHER_MODEL, trust_remote_code=True)
    # transformers >= 5 đổi tên kwarg dtype; distiller.py xử lý y hệt.
    dtype_kw = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"
    teacher = AutoModel.from_pretrained(
        TEACHER_MODEL, trust_remote_code=True, **{dtype_kw: torch.bfloat16}
    ).to(DEVICE).eval()
    out = encode_texts(
        teacher, tok_teacher, PROBE_TEXTS, device=DEVICE, pooling=TEACHER_POOLING,
        batch_size=64, max_length=PROBE_MAX_LENGTH, layers=False, amp=False, progress=True,
    )
    torch.save(
        {
            "embeddings": out["final"],            # [N, d_T] float32, CHƯA normalize
            "probe_digest": PROBE_DIGEST,
            "teacher_model_name": TEACHER_MODEL,
            "pooling_method": TEACHER_POOLING,
            "max_length": PROBE_MAX_LENGTH,
        },
        PROBE_TEACHER_PATH,
    )
    del teacher
    torch.cuda.empty_cache()
    print(f"[probe] lưu {PROBE_TEACHER_PATH.name}: {tuple(out['final'].shape)}")

STUDENT_INIT_PATH = PROBE_DIR / f"student_init_{slug(STUDENT_MODEL)}_{STUDENT_POOLING}_{PROBE_DIGEST}.pt"
if STUDENT_INIT_PATH.is_file():
    print(f"[probe] student init đã có: {STUDENT_INIT_PATH.name}")
else:
    print(f"[probe] encode student chưa train {STUDENT_MODEL}...")
    tok_student = AutoTokenizer.from_pretrained(STUDENT_MODEL)
    student = load_student(STUDENT_MODEL, None, DEVICE)
    full = encode_texts(student, tok_student, PROBE_TEXTS, device=DEVICE, pooling=STUDENT_POOLING,
                        batch_size=PROBE_BATCH_SIZE, max_length=PROBE_MAX_LENGTH, layers=False, progress=True)
    core = encode_texts(student, tok_student, PROBE_CORE_TEXTS, device=DEVICE, pooling=STUDENT_POOLING,
                        batch_size=PROBE_BATCH_SIZE, max_length=PROBE_MAX_LENGTH, layers=True, progress=True)
    torch.save(
        {
            "final": full["final"].to(torch.float16),   # [N, d_S] step 0
            "layers": core["layers"],                   # [L+1, N_core, d_S] step 0
            "core_index": torch.from_numpy(CORE_INDEX),
            "probe_digest": PROBE_DIGEST,
            "student_model_name": STUDENT_MODEL,
            "pooling": STUDENT_POOLING,
        },
        STUDENT_INIT_PATH,
    )
    del student
    torch.cuda.empty_cache()
    print(f"[probe] lưu {STUDENT_INIT_PATH.name}")


In [ ]:
# 5. Arm registry: mọi arm của protocol, nhóm theo claim, và command của từng arm.
#
# Định nghĩa flag của các arm target-map lấy từ scripts/run_target_map_ablation.py
# (một nguồn duy nhất, script và notebook không lệch nhau). Command dựng theo đúng
# khuôn của main_tables.ipynb, cộng thêm:
#   --seed             seed train của arm
#   --save_every 1     checkpoint + weight student MỖI epoch (nguồn của probe dump)
#   --weights_dir      weight student gọn (không optimizer) vào <arm>/weights/
#   matched HP         cùng lr/batch/epoch/λ cho mọi arm (§0.3)
# Arm thiếu code có needs != None: không có command, bị bỏ qua khi chạy nhưng vẫn
# nằm trong plan để bảng tổng hợp nói rõ nó chưa có.
import shlex

from scripts import run_target_map_ablation as ablation

# Cache teacher nằm NGOÀI run, dùng chung cho mọi run của cặp (khoá theo teacher,
# pooling, max_length và *nội dung* corpus, nên corpus dedup có cache riêng).
CACHE_DIR = OUTPUT_BASE / "teacher_cache"
CACHED_TEACHER_METHODS = ("talas", "geoode", "rkd")
ARMS_ROOT = RUN_ROOT / "arms"


def build_command(name, method, train_data, seed, flags=()):
    """Command của một arm; `name` là tên thư mục output."""
    arm_dir = ARMS_ROOT / name
    command = [
        sys.executable,
        str(PROJECT_DIR / "main.py"),
        "--method", method,
        "--train_data", str(train_data),
        "--student_model", STUDENT_MODEL,
        # Teacher truyền cho MỌI method, kể cả simcse (control không distill: distiller
        # không tải teacher nhưng config ghi lại nó là control của cặp nào).
        "--teacher_model", TEACHER_MODEL,
        "--teacher_pooling", TEACHER_POOLING,
        "--batch_size", str(MATCHED_HP["batch_size"]),
        "--epochs", str(MATCHED_HP["epochs"]),
        "--save_every", "1",
        "--lr", str(MATCHED_HP["learning_rate"]),
        "--max_length", str(MATCHED_HP["max_length"]),
        "--seed", str(seed),
        "--save_dir", str(arm_dir),
        "--weights_dir", str(arm_dir / "weights"),
        "--num_workers", str(NUM_WORKERS),
        "--pair_threshold_source", PAIR_THRESHOLD_SOURCE,
        "--eval_every", str(EVAL_EVERY),
        "--no_wandb",
    ]
    if EVAL_ON_TEST_EACH_EPOCH:
        command.append("--evaluate_test_each_epoch")
    if not EVAL_RETRIEVAL:
        command.append("--no_eval_retrieval")
    if method in CACHED_TEACHER_METHODS:
        command.extend(["--cache_dir", str(CACHE_DIR)])
    flags = list(flags)
    if method == "geoode" and "--lambda_ctr" not in flags:
        command.extend(["--lambda_ctr", str(MATCHED_HP["lambda_ctr"])])
    # Flag riêng của arm đi sau cùng để thắng mọi default.
    command.extend(flags)
    return command


def fixed_interface(subspace, gauge, draw=None):
    """Settings của một arm map đóng băng, theo đúng bảng arm của script."""
    settings = {**ablation.SUBSPACE_ARMS[subspace], **ablation.GAUGE_ARMS[gauge]}
    if subspace in ablation.STOCHASTIC_SUBSPACES:
        settings["projection_seed"] = int(draw or 0)
    if gauge == "random":
        settings["gauge_random_seed"] = int(draw or 0)
    return settings


ARM_SPECS = []


def add(name, group, *, method="geoode", settings=None, flags=(), dataset=None, note="", needs=None):
    if not GROUPS.get(group, False):
        return
    ARM_SPECS.append(
        {
            "name": name,
            "group": group,
            "method": method,
            "settings": settings,
            "flags": list(flags),
            "dataset": dataset or DATASET,
            "note": note,
            "needs": needs,
        }
    )


# ---- C1: the interface (Table 2, Figures 2, 6, 7, 8) ---------------------------
add("pca__procrustes", "C1", settings=fixed_interface("pca", "procrustes"),
    note="ours: PCA -> d_S, một Procrustes R về student init, đóng băng")
add("pca__none", "C1", settings=fixed_interface("pca", "none"),
    note="PCA only (sbert <= 5.4, HPD teacher side): gauge có làm gì không?")
for draw in range(DRAWS):
    add(f"pca__random__d{draw}", "C1", settings=fixed_interface("pca", "random", draw),
        note="PCA + xoay Haar cùng chi phí: null band của G")
for draw in range(DRAWS):
    add(f"random__none__d{draw}", "C1", settings=fixed_interface("random", "none", draw),
        note="subspace Haar orthonormal cùng rank: không có phổ")
add("mrl_prefix__none", "C1", settings=fixed_interface("mrl_prefix", "none"),
    note="d_S toạ độ đầu của teacher (Matryoshka prefix)")
for scale in LEARNED_LR_SCALES:
    add(f"learned_t2s__lr{scale:g}", "C1",
        settings={"projection_type": "learned_t2s", "gauge_align": False, "learned_projector_lr_scale": scale},
        note="W in R^{d_T x d_S} học cùng student (EMO, sbert v5.5)")
for scale in LEARNED_LR_SCALES:
    add(f"learned_s2t__lr{scale:g}", "C1",
        settings={"projection_type": "learned_s2t", "gauge_align": False, "learned_projector_lr_scale": scale},
        note="W in R^{d_S x d_T} nâng student lên không gian teacher (TALAS, LEAF)")
add("procrustes_per_batch", "C1",
    needs="Procrustes giải lại closed-form theo từng batch trong criterion (EdgePoint2) — chưa có code (~40 dòng)")
add("pca__mse", "C1", settings=fixed_interface("pca", "none"),
    flags=["--endpoint_loss", "mse", "--lambda_ctr", "0"],
    note="recipe sentence-transformers <= 5.4: PCA target + MSE, không InfoNCE, không gauge")
add("simcse_only", "C1", method="simcse", note="không teacher: floor của mọi ladder")
for k in (64, 128, 256):
    add(f"pca_k{k}__procrustes", "C1",
        needs=f"--projection_rank {k}: target PCA-{k} pad 0 tới d_S (Figure 7, distortion-vs-score) — chưa có code")

# ---- C2: structure for free (Figure 3, 9-11) -----------------------------------
add("ours+gram", "C2", needs="--lambda_gram: term pairwise-similarity (RKD/SP) ở layer cuối, trên nền ours — chưa có code")
add("ours+lasd", "C2", needs="--lambda_lasd: relational self-distillation giữa layer kề nhau (TALAS-style) — chưa có code")
add("ours+anchor_k", "C2", needs="--anchor_layers k: cosine anchoring k layer trên cùng (TALAS-style) — chưa có code")

# ---- C3: depth (Figure 4, 12; Table 5) ------------------------------------------
add("freeze_lower", "C3", needs="--freeze_lower_layers N: chỉ train 2 block trên cùng (~15 dòng ở setup_training) — chưa có code")

# ---- G: gauge as an optimisation effect (Figure 5; Table 6) --------------------
# Null band = pca__random__d* ở C1. Hai control còn lại cần code ở target map:
for theta in (0.25, 0.5, 0.75):
    add(f"gauge_theta{theta:g}", "G",
        needs=f"--gauge_theta {theta}: Q(θ) = R exp(θ log(Rᵀ Q_rand)) nội suy giữa Procrustes và Haar — chưa có code")
add("gauge_rank_one", "G",
    needs="--gauge_rotation rank_one: Householder đưa mean target lên mean student init — chưa có code")

# ---- M: main results & external validity (Table 1, 3, 7; Figure 13) ------------
add("talas__matched", "M", method="talas", note="TALAS ở matched HP (optimizer SAM như code gốc)")
add("talas__adamw", "M", method="talas", needs="--talas_optimizer adamw: TALAS bỏ SAM (§0.3 a) — chưa có code")
add("ours__talas15k", "M", settings=fixed_interface("pca", "procrustes"), dataset=TALAS_SMALL_DATASET,
    note="(§0.3 b) ours trên corpus nhỏ của TALAS")
for key in CORPUS_SWEEP_DATASETS:
    add(f"ours__{key}", "M", settings=fixed_interface("pca", "procrustes"), dataset=key, note="corpus-size sweep")
    add(f"talas__{key}", "M", method="talas", dataset=key, note="corpus-size sweep, baseline mạnh nhất")

# Nhân theo seed: tên mang __s<seed> khi có nhiều seed, để hai seed không ghi đè nhau.
ARM_PLAN = []
for spec in ARM_SPECS:
    for seed in SEEDS:
        arm = dict(spec, base=spec["name"], seed=seed)
        arm["name"] = f"{spec['name']}__s{seed}" if len(SEEDS) > 1 else spec["name"]
        arm["train_data"] = str(TRAIN_DATA_BY_DATASET[spec["dataset"]])
        ARM_PLAN.append(arm)

ARM_COMMANDS = {}
for arm in ARM_PLAN:
    if arm["needs"] is not None:
        continue
    flags = ablation.arm_flags(arm["settings"]) if arm["settings"] else []
    flags += arm["flags"]
    ARM_COMMANDS[arm["name"]] = build_command(arm["name"], arm["method"], arm["train_data"], arm["seed"], flags)

runnable = [a for a in ARM_PLAN if a["needs"] is None]
blocked = [a for a in ARM_PLAN if a["needs"] is not None]
print(f"{len(ARM_PLAN)} arm trong plan: {len(runnable)} chạy được, {len(blocked)} thiếu code -> {ARMS_ROOT}\n")
for arm in ARM_PLAN:
    if arm["needs"] is None:
        print(f"[{arm['group']}] {arm['name']}: {shlex.join(ARM_COMMANDS[arm['name']])}\n")
    else:
        print(f"[{arm['group']}] {arm['name']}: THIẾU CODE — {arm['needs']}\n")


In [ ]:
# 6. Chạy tuần tự mọi arm; sau mỗi arm, dump probe (§0.1) và cắt checkpoint nặng.
import shutil
import time

from src.structural_audit import encode_texts, load_student

PROGRESS_EVERY_SEC = 30
PROGRESS_MAX_CHARS = 160


def has_final_test(metrics_path):
    """True chỉ với record cuối run: có "test", không có "train"."""
    if not metrics_path.is_file():
        return False
    with metrics_path.open(encoding="utf-8") as handle:
        return any(
            json.loads(line).get("test") and not json.loads(line).get("train")
            for line in handle
            if line.strip()
        )


def stream_output(stream, log_handle):
    """Tee output của subprocess: log đầy đủ ra file, notebook chỉ hiện progress mới nhất."""
    buffer = ""
    last_progress = 0.0
    progress_shown = False
    while True:
        chunk = stream.read(4096)
        if not chunk:
            break
        buffer += chunk
        parts = buffer.replace("\r\n", "\n").replace("\r", "\n").split("\n")
        buffer = parts.pop()
        for line in parts:
            log_handle.write(line + "\n")
            if "%|" in line:
                now = time.perf_counter()
                if now - last_progress >= PROGRESS_EVERY_SEC:
                    print("\r" + line[:PROGRESS_MAX_CHARS].ljust(PROGRESS_MAX_CHARS), end="", flush=True)
                    last_progress = now
                    progress_shown = True
            elif line.strip():
                if progress_shown:
                    print()
                    progress_shown = False
                print(line)
        log_handle.flush()
    if buffer:
        log_handle.write(buffer + "\n")
        if "%|" not in buffer:
            print(buffer)
    if progress_shown:
        print()


def read_records(path):
    if not path.is_file():
        return []
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def write_train_summary(arm, arm_dir):
    """train_summary.json: loss cuối, điểm test cuối, metadata của target map / map học."""
    records = read_records(arm_dir / "metrics.jsonl")
    train_epochs = [r["train"] for r in records if r.get("train")]
    final = next((r["test"] for r in records if r.get("test") and not r.get("train")), None)
    summary = {
        "arm": arm["name"], "base": arm["base"], "group": arm["group"], "method": arm["method"],
        "seed": arm["seed"], "dataset": arm["dataset"], "train_data": arm["train_data"],
        "settings": arm["settings"], "flags": arm["flags"], "note": arm["note"],
        "matched_hp": MATCHED_HP, "probe_digest": PROBE_DIGEST,
        "train_by_epoch": train_epochs,
        "final_train": train_epochs[-1] if train_epochs else None,
        "final_test_summary": final["summary"] if final else None,
        "final_test": {k: final[k] for k in ("classification", "pair", "sts", "retrieval")} if final else None,
    }
    projection_path = arm_dir / "teacher_projection.pt"
    if projection_path.is_file():
        saved = torch.load(projection_path, map_location="cpu", weights_only=False)
        summary["projection"] = {
            k: saved.get(k) for k in (
                "projection_type", "projection_seed", "pca_center_fit", "pca_subtract_mean",
                "explained_energy", "gauge_align", "gauge_rotation", "gauge_random_seed",
                "student_dim", "teacher_dim",
            )
        }
        summary["gauge_stats"] = saved.get("gauge_stats")
    # Map học: W của từng epoch được rút ra khỏi checkpoint (nhỏ) trước khi checkpoint bị cắt.
    learned = sorted(arm_dir.glob("checkpoint_epoch_*.pt"), key=lambda p: int(p.stem.split("_")[-1]))
    maps = {}
    for path in learned:
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
        state = checkpoint.get("criterion_state_dict") or {}
        weight = next((v for k, v in state.items() if k.endswith("linear.weight")), None)
        if weight is None:
            break
        epoch = int(path.stem.split("_")[-1])
        maps[epoch] = weight.float()
    if maps:
        torch.save({"epochs": sorted(maps), "weight": maps, "note": "nn.Linear weight [out, in]"},
                   arm_dir / "learned_map.pt")
        summary["learned_map"] = {"epochs": sorted(maps), "shape": list(next(iter(maps.values())).shape)}
    (arm_dir / "train_summary.json").write_text(json.dumps(summary, indent=2, default=float), encoding="utf-8")
    return summary


def dump_probe(arm, arm_dir, force=False):
    """probe_final.pt (mọi epoch, cả probe set) + probe_layers.pt (mọi layer, core) + train_summary.json."""
    final_path, layers_path = arm_dir / "probe_final.pt", arm_dir / "probe_layers.pt"
    if final_path.is_file() and layers_path.is_file() and (arm_dir / "train_summary.json").is_file() and not force:
        return "exists"
    weights_dir = arm_dir / "weights"
    epochs = sorted(int(p.stem.split("_")[-1]) for p in weights_dir.glob("student_epoch_*.pt"))
    if not epochs:
        raise FileNotFoundError(f"{arm['name']}: không có weights/student_epoch_*.pt để dump probe")
    init = torch.load(STUDENT_INIT_PATH, map_location="cpu", weights_only=False)
    assert init["probe_digest"] == PROBE_DIGEST
    finals = {0: init["final"]}
    layers = {0: init["layers"]}
    tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
    for epoch in epochs:
        model = load_student(STUDENT_MODEL, weights_dir / f"student_epoch_{epoch}.pt", DEVICE)
        full = encode_texts(model, tokenizer, PROBE_TEXTS, device=DEVICE, pooling=STUDENT_POOLING,
                            batch_size=PROBE_BATCH_SIZE, max_length=PROBE_MAX_LENGTH, layers=False)
        finals[epoch] = full["final"].to(torch.float16)
        if PROBE_LAYERS_EVERY_EPOCH or epoch == epochs[-1]:
            core = encode_texts(model, tokenizer, PROBE_CORE_TEXTS, device=DEVICE, pooling=STUDENT_POOLING,
                                batch_size=PROBE_BATCH_SIZE, max_length=PROBE_MAX_LENGTH, layers=True)
            layers[epoch] = core["layers"]
        del model
        torch.cuda.empty_cache()
        print(f"  [probe] {arm['name']} epoch {epoch}/{epochs[-1]} encoded", flush=True)
    meta = {"probe_digest": PROBE_DIGEST, "student_model_name": STUDENT_MODEL, "pooling": STUDENT_POOLING,
            "max_length": PROBE_MAX_LENGTH, "arm": arm["name"], "epochs": [0, *epochs]}
    torch.save({**meta, "final": finals}, final_path)
    torch.save({**meta, "epochs": sorted(layers), "core_index": torch.from_numpy(CORE_INDEX), "layers": layers}, layers_path)
    write_train_summary(arm, arm_dir)
    return "dumped"


def prune_checkpoints(arm_dir):
    """Cắt checkpoint đầy đủ (optimizer state) theo KEEP_FULL_CHECKPOINTS; weights/ và probe luôn giữ."""
    if KEEP_FULL_CHECKPOINTS == "all":
        return 0
    checkpoints = sorted(arm_dir.glob("checkpoint_epoch_*.pt"), key=lambda p: int(p.stem.split("_")[-1]))
    doomed = list(checkpoints) if KEEP_FULL_CHECKPOINTS == "none" else checkpoints[:-1]
    doomed += [p for p in [arm_dir / "best_model.pt"] if p.is_file()]
    freed = sum(p.stat().st_size for p in doomed)
    for path in doomed:
        path.unlink()
    return freed


env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
env["TOKENIZERS_PARALLELISM"] = "false"
env["WANDB_MODE"] = "disabled"
env["TQDM_MININTERVAL"] = str(PROGRESS_EVERY_SEC)

ARMS_ROOT.mkdir(parents=True, exist_ok=True)
(RUN_ROOT / "run_config.json").write_text(
    json.dumps(
        {
            "run_name": RUN_NAME, "pair": PAIR, "pair_config": PAIR_CONFIG,
            "student_model": STUDENT_MODEL, "teacher_model": TEACHER_MODEL,
            "teacher_pooling": TEACHER_POOLING, "student_pooling": STUDENT_POOLING,
            "dataset": DATASET, "dataset_tag": DATASET_TAG, "dedup": DEDUP_CORPUS,
            "train_data": str(TRAIN_DATA), "train_data_by_dataset": {k: str(v) for k, v in TRAIN_DATA_BY_DATASET.items()},
            "dedup_manifests": DEDUP_MANIFESTS, "matched_hp": MATCHED_HP, "seeds": SEEDS, "draws": DRAWS,
            "learned_lr_scales": LEARNED_LR_SCALES, "groups": GROUPS, "eval_retrieval": EVAL_RETRIEVAL,
            "probe_path": str(PROBE_PATH), "probe_digest": PROBE_DIGEST,
            "probe_teacher_path": str(PROBE_TEACHER_PATH), "student_init_path": str(STUDENT_INIT_PATH),
            "cache_dir": str(CACHE_DIR), "arms_root": str(ARMS_ROOT),
            "arms": [{k: v for k, v in arm.items()} for arm in ARM_PLAN],
        },
        indent=2, default=str,
    ),
    encoding="utf-8",
)

run_status = []
for position, arm in enumerate(ARM_PLAN, start=1):
    name = arm["name"]
    arm_dir = ARMS_ROOT / name
    if arm["needs"] is not None:
        print(f"[NEEDS CODE] {name}: {arm['needs']}")
        run_status.append({"arm": name, "status": "needs_code", "seconds": 0.0})
        continue
    metrics_path = arm_dir / "metrics.jsonl"
    log_path = ARMS_ROOT / f"{name}_train.log"
    if has_final_test(metrics_path):
        state = dump_probe(arm, arm_dir)
        prune_checkpoints(arm_dir)
        print(f"[SKIP] {name} đã có final test (probe: {state})")
        run_status.append({"arm": name, "status": "skipped_complete", "seconds": 0.0})
        continue
    if metrics_path.exists():
        raise RuntimeError(f"{name} có run dở dang tại {arm_dir}. Xoá thư mục đó hoặc dùng RUN_NAME mới.")

    print("\n" + "#" * 88)
    print(f"ARM {position}/{len(ARM_PLAN)} [{arm['group']}]: {name} — {arm['note']}")
    print(f"Log: {log_path}")
    print("#" * 88)
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            ARM_COMMANDS[name], cwd=PROJECT_DIR, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        assert process.stdout is not None
        stream_output(process.stdout, log_handle)
        return_code = process.wait()
    elapsed = time.perf_counter() - started
    status = "complete" if return_code == 0 and has_final_test(metrics_path) else "failed"
    if status == "complete":
        dump_probe(arm, arm_dir)
        freed = prune_checkpoints(arm_dir)
        print(f"  [prune] giải phóng {freed / 2**30:.1f} GiB checkpoint")
    run_status.append({"arm": name, "status": status, "seconds": elapsed})
    print(f"[{status.upper()}] {name} in {elapsed / 60:.1f} minutes")
    if status == "failed" and STOP_ON_ERROR:
        raise RuntimeError(f"Arm {name} failed; xem log tại {log_path}")

print("\nRun status:")
for item in run_status:
    print(f"  {item['arm']:28s} {item['status']:18s} {item['seconds'] / 60:8.1f} min")
(RUN_ROOT / "run_status.json").write_text(json.dumps(run_status, indent=2), encoding="utf-8")


In [ ]:
# 7. Bổ sung: dump probe cho arm nào đã xong mà còn thiếu (ví dụ run từ session cũ).
#
# Idempotent: arm đã có đủ ba file bị bỏ qua. Đặt FORCE_REDUMP = True để dump lại
# toàn bộ (ví dụ sau khi đổi PROBE_MAX_LENGTH — nhớ rằng probe set thì không đổi).
FORCE_REDUMP = False

for arm in ARM_PLAN:
    arm_dir = ARMS_ROOT / arm["name"]
    if arm["needs"] is not None or not has_final_test(arm_dir / "metrics.jsonl"):
        continue
    state = dump_probe(arm, arm_dir, force=FORCE_REDUMP)
    freed = prune_checkpoints(arm_dir)
    print(f"{arm['name']:28s} probe={state:7s} freed={freed / 2**30:.1f} GiB")


In [ ]:
# 8. Tổng hợp: điểm test cuối, loss cuối và metadata target map của từng arm.
import pandas as pd
from IPython.display import display

SUMMARY_KEYS = ("avg_iod", "avg_ood", "avg_retrieval", "avg_all")
rows = []
for arm in ARM_PLAN:
    arm_dir = ARMS_ROOT / arm["name"]
    row = {"arm": arm["name"], "base": arm["base"], "group": arm["group"], "method": arm["method"],
           "seed": arm["seed"], "dataset": arm["dataset"],
           "status": "needs_code" if arm["needs"] else ("done" if (arm_dir / "train_summary.json").is_file() else "missing")}
    if row["status"] == "done":
        summary = json.loads((arm_dir / "train_summary.json").read_text(encoding="utf-8"))
        test = summary.get("final_test_summary") or {}
        row.update({k: (None if test.get(k) is None else float(test[k])) for k in SUMMARY_KEYS})
        train = summary.get("final_train") or {}
        row.update({f"final_{k}": train.get(k) for k in ("loss", "loss_end", "loss_ctr", "cos_final")})
        projection = summary.get("projection") or {}
        row["projection_type"] = projection.get("projection_type")
        row["explained_energy"] = projection.get("explained_energy")
        gauge = summary.get("gauge_stats") or {}
        row["participation_ratio"] = gauge.get("participation_ratio")
        row["cos_init_after_gauge"] = gauge.get("cos_after")
    rows.append(row)
arms_summary = pd.DataFrame(rows)
arms_summary.to_csv(RUN_ROOT / "arms_summary.csv", index=False)
pd.DataFrame(run_status).to_csv(RUN_ROOT / "run_status.csv", index=False)

print("ARMS")
display(arms_summary.style.format(precision=4, na_rep="-"))
done = arms_summary[arms_summary["status"] == "done"]
if not done.empty:
    print("\nMEAN ± STD OVER SEEDS (avg_all)")
    display(done.groupby(["group", "base"], sort=False)["avg_all"].agg(["mean", "std", "count"]).style.format(precision=4, na_rep="-"))
print(f"\nSaved: {RUN_ROOT / 'arms_summary.csv'}")
print("Phân tích ladder/depth/figure: mở notebooks/audit_analysis_qwen_minilm.ipynb với RUN_NAME =", RUN_NAME)


In [ ]:
# 9. Tuỳ chọn: đóng gói phần nhẹ của run (bảng, log, probe dump, train_summary) — không gồm weights/.
import shutil

ARCHIVE_DIR = RUN_ROOT.parent / f"{RUN_NAME}_light"
if ARCHIVE_DIR.exists():
    shutil.rmtree(ARCHIVE_DIR)
for path in RUN_ROOT.rglob("*"):
    if path.is_dir() or "weights" in path.parts or path.name.startswith("checkpoint_epoch_") or path.name == "best_model.pt":
        continue
    target = ARCHIVE_DIR / path.relative_to(RUN_ROOT)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, target)
archive_path = shutil.make_archive(str(ARCHIVE_DIR), "zip", root_dir=ARCHIVE_DIR)
shutil.rmtree(ARCHIVE_DIR)
print(f"Created archive: {archive_path}")
if IN_COLAB and not SAVE_TO_GOOGLE_DRIVE:
    from google.colab import files

    files.download(archive_path)
